## State estimation

The purpose of state estimation is to reduce the uncertainty of the physical world to which our robot is subjected. During its treasure hunt, our Thymio robot evolves by detecting its environment and performing actions based on mathematical representations of that environment. Although this environment appears relatively straightforward, it is inherently unpredictable. This is partly because the robot can only perceive it through imperfect sensors that are difficult to model due to the complexity of their manufacturing processes. However, the environment is also unpredictable due to its lack of stability over time.

Therefore, we need a model that can filter out this uncertainty, enabling our robot to perform its tasks with greater robustness. However, in order to select such a model, we must first define the state-space representation of our system.

### Treasure map recall

<div style="text-align: center;">
    <img src="./imgs/map_drawing.jpeg" alt="Treasure map" width="800"/>
</div>

### State-space representation

#### State variable definition

As we have seen, the state variable is the smallest possible subset of variables required to describe the evolution of the robot over time. It means that it should contain only the important quantities that are affecting the robot state and no redundant informations. For this project, we'll define the state of the Thymio as follow:

- Thymio is a differential dirve robot (DDR) with 3 DOFs, so three variables are required to describe its pose: $x$, $y$ and $\theta$

- As explained in [1-Components of a Mobile Robot, slide 11](https://moodle.epfl.ch/pluginfile.php/2703472/mod_resource/content/13/Slides%2001%20-%20Components%20of%20a%20Mobile%20Robot.pdf), its dynamics can be represented solely using the norm of its forward and angular speed. We will denote these with $v$ and $\omega$, and their relationship with the motor speeds is defined by the Astolfi controller as follows:

```math
\begin{equation*}
    \begin{align*}
        v &= \frac{r\dot{\phi}_r}{2} + \frac{r\dot{\phi}_l}{2} \\
        \omega &= \frac{r\dot{\phi}_r}{d}-\frac{r\dot{\phi}_l}{d}
    \end{align*}
\end{equation*}
```

Where $\dot{\phi}_i$ is the motor speed, $r$ the wheel radius and $d$ the distance between both wheels.

Hence, the complete state variable will be denoted as $\vec{x}(t) =
\begin{pmatrix}
  x\\ 
  y\\
  \theta\\
  v\\
  \omega
\end{pmatrix}$

#### State-space equations

Expressing the Thymio’s state in a fixed reference frame relative to the map leads to a nonlinear system, because the robot’s velocity must be projected onto the reference axes using trigonometric (sine and cosine) terms. To map the velocities from the robot's referential frame to the global reference frame of the map, we use rotational matrices as we have seen in [1-Components of a Mobile Robot, slide 12](https://moodle.epfl.ch/pluginfile.php/2703472/mod_resource/content/13/Slides%2001%20-%20Components%20of%20a%20Mobile%20Robot.pdf):

- To align both x axis, we perform a rotation around the z axis of the global frame, using:

```math
\begin{equation*}
    R_z(-\theta) = R_z(\theta)^{-1} =
    \begin{pmatrix}
        cos(\theta) & sin(\theta) & 0 \\
        -sin(\theta) & cos(\theta) & 0 \\
        0 & 0 & 1
    \end{pmatrix}
\end{equation*}
```

- Then to reverse the z axis, we perform a rotation around the x axis of the global frame, using:

```math
\begin{equation*}
    R_x(\pi) =
    \begin{pmatrix}
        1 & 0 & 0 \\
        0 & -1 & 0 \\
        0 & 0 & -1
    \end{pmatrix}
\end{equation*}
```

- Both rotations are illustrated bellow:

<div style="text-align: center;">
    <img src="./imgs/referential_rotation.jpeg" alt="Treasure map" width="800"/>
</div>

Therefore, the velocities of the robot in the global referential frame are:

```math
\begin{equation*}
    \begin{pmatrix}
        \dot{x} \\
        \dot{y} \\
        \dot{\theta}
    \end{pmatrix} = R_x(\pi)R_z(-\theta)
    \begin{pmatrix}
        v \\
        0 \\
        \omega
    \end{pmatrix} = 
    \begin{pmatrix}
        cos(\theta) & sin(\theta) & 0 \\
        sin(\theta) & -cos(\theta) & 0 \\
        0 & 0 & -1
    \end{pmatrix}
    \begin{pmatrix}
        v \\
        0 \\
        \omega
    \end{pmatrix} =
    \begin{pmatrix}
        vcos(\theta) \\
        vsin(\theta) \\
        -\omega
    \end{pmatrix}
\end{equation*}
```

And we can use then to define the discrete state-space equations:

```math
\begin{equation*}
    \begin{align*}
        \vec{x}_{k} &= 
            \begin{pmatrix}
                x_{k}\\ 
                y_{k}\\
                \theta_{k}\\
                v_{k}\\
                \omega_{k}
            \end{pmatrix} 
            = \boldsymbol{g}(\vec{x}_{k-1}, \vec{u}_k) = 
            \begin{pmatrix}
                x_{k-1} + dt \cdot v_{k-1} \cdot cos(\theta_{k-1})\\ 
                y_{k-1} + dt \cdot v_{k-1} \cdot sin(\theta_{k-1})\\
                \theta_{k-1} - dt \cdot \omega_{k-1}\\
                \frac{r\dot{\phi}_r}{2} + \frac{r\dot{\phi}_l}{2}\\
                \frac{r\dot{\phi}_r}{d}-\frac{r\dot{\phi}_l}{d}
            \end{pmatrix} =
            \begin{pmatrix}
                x_{k-1} + dt \cdot v_{k-1} \cdot cos(\theta_{k-1})\\ 
                y_{k-1} + dt \cdot v_{k-1} \cdot sin(\theta_{k-1})\\
                \theta_{k-1} - dt \cdot \omega_{k-1}\\
                \frac{\lambda}{2} (u_{rk} + u_{lk})\\
                \frac{\lambda}{d} (u_{rk} - u_{lk})
            \end{pmatrix} \quad \text{with} \quad \vec{u}_k =
                \begin{pmatrix}
                    u_{rk} \\
                    u_{lk}
                \end{pmatrix} \\
        \vec{z}_k &= 
            \begin{pmatrix}
                x_{k,measured} \\
                y_{k,measured} \\
                \theta_{k,measured} \\
                u_{r,k,measured}\\
                u_{l,k,measured}
            \end{pmatrix} = \boldsymbol{h}(\vec{x}_{k}) =
            \begin{pmatrix}
                x_k \\
                y_k \\
                \theta_k \\
                \frac{1}{\lambda}(v_k + \frac{d}{2}\omega_k)\\
                \frac{1}{\lambda}(v_k - \frac{d}{2}\omega_k)
            \end{pmatrix}
    \end{align*}
\end{equation*}
```

Where $dt$ is the sampling period and $\lambda$ a conversion factor to convert motor command into a speed in $mm/s$.

Now that we have defined the state-space representation of our system, we need to estimate the type of noise that will affect it and select an appropriate uncertainty filter algorithm from those discussed in class.

### Noise estimation